# Generate Popularity-Enriched QA Datasets

This notebook processes three QA datasets (PopQA, Natural Questions, TriviaQA) and enriches them with Wikipedia popularity metrics.

**Output Files:**
- `popqa_with_popularity.parquet`
- `nq_with_popularity.parquet`
- `triviaqa_with_popularity.parquet`

**Output Schema:**
- `question_id`: Unique identifier for the question
- `question_text`: The question text
- `answer_texts`: List of answer strings
- `wikipedia_id`: Wikipedia page ID (used for matching across all datasets)
- `wikipedia_title`: Wikipedia page title
- `popularity_avg`: Average monthly pageviews (Jan 2022 - Jan 2023)
- `popularity_rank`: Popularity rank (from rank_avg in source dataset)

## 1. Imports and Configuration

In [1]:
import gc
import json
import os
import pandas as pd
from datasets import load_dataset
from config import DATA_DIR, CACHE_DIR, ROOT_DIR
from tqdm import tqdm
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Configure output paths
OUTPUT_DIR = Path(DATA_DIR)
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

POPQA_OUTPUT = OUTPUT_DIR / "popqa_with_popularity.parquet"
NQ_OUTPUT = OUTPUT_DIR / "nq_with_popularity.parquet"
TRIVIAQA_OUTPUT = OUTPUT_DIR / "triviaqa_with_popularity.parquet"
HOTPOTQA_OUTPUT = OUTPUT_DIR / "hotpotqa_with_popularity.parquet"
FEVER_OUTPUT = OUTPUT_DIR / "fever_with_popularity.parquet"
TREX_OUTPUT = OUTPUT_DIR / "trex_with_popularity.parquet"

# Dataset paths
HUGGINGFACE_POP_WIKI = "Cyro1/enwiki_pageviews_m"
POPQA_PATH = "akariasai/PopQA"
NQ_PATH = "facebook/kilt_tasks"
TRIVIAQA_PATH = "facebook/kilt_tasks"
HOTPOTQA_PATH = "facebook/kilt_tasks"
FEVER_PATH = "facebook/kilt_tasks"
TREX_PATH = "facebook/kilt_tasks"


# HuggingFace upload configuration
HUGGINGFACE_TOKEN = os.getenv("HUGGINGFACE_TOKEN")
HUGGINGFACE_UPLOAD_REPO = "Cyro1/popularity-enriched-qa-datasets"
HUGGINGFACE_UPLOAD_SUBSETS = {
    "PopQA": ("pop_qa", POPQA_OUTPUT),
    "NaturalQuestions": ("natural_questions", NQ_OUTPUT),
    "TriviaQA": ("trivia_qa", TRIVIAQA_OUTPUT),
    "HotPotQA": ("hotpot_qa", HOTPOTQA_OUTPUT),
    "FEVER": ("fever", FEVER_OUTPUT),
    "TREX": ("trex", TREX_OUTPUT),
}

README_TEXT = """# Popularity-Enriched QA Datasets

This dataset repo hosts popularity-enriched versions of PopQA, Natural Questions, and TriviaQA.
Each subset retains the enrichment schema produced by this notebook (question + pron and popularity metrics).

## Subsets

- `pop_qa`: Popularity-enriched PopQA test split
- `natural_questions`: Wikipedia-provenance Natural Questions validation set
- `trivia_qa`: TriviaQA validation subset matched to KILT and original TriviaQA IDs
- `hotpot_qa`: HotPotQA validation subset matched to KILT and original HotPotQA IDs

## Schema

- `question_id`: question identifier
- `question_text`: raw question text
- `answer_texts`: list of candidate answers
- `wikipedia_id`: Wikipedia provenance page id
- `wikipedia_title`: page title
- `popularity_avg`: average monthly pageviews
- `popularity_rank`: rank derived from the popularity source

## Loading

Use `datasets.load_dataset("Cyro1/popularity-enriched-qa-datasets", split="popqa")` to stream the PopQA subset and swap `split` for each subset name.
"""

print("Configuration loaded successfully!")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Cache directory: {CACHE_DIR}")
print(f"HuggingFace repo: {HUGGINGFACE_UPLOAD_REPO}")
print("README text will be published directly to the HuggingFace dataset repo.")

/Users/cyro/Documents/VSC/PopularityBias/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Configuration loaded successfully!
Output directory: /Users/cyro/Documents/VSC/PopularityBias/data
Cache directory: /Users/cyro/Documents/VSC/PopularityBias/data/cache
HuggingFace repo: Cyro1/popularity-enriched-qa-datasets
README text will be published directly to the HuggingFace dataset repo.


## 2. Load Base Datasets

Load Wikipedia popularity data that will be joined with all QA datasets.

In [2]:
print("Loading Wikipedia popularity data...")
pop_ds = load_dataset(
    HUGGINGFACE_POP_WIKI,
    split="train+test",
    cache_dir=CACHE_DIR
)

# Convert to DataFrame and clean
pop_df = (
    pop_ds
    .select_columns(["wikipedia_id", "wikipedia_title", "popularity_avg", "rank_avg"])
    .to_pandas()
    .rename(columns={"rank_avg": "popularity_rank"})
)

# Clean titles
pop_df["wikipedia_title"] = pop_df["wikipedia_title"].str.strip()
pop_df = pop_df[pop_df["popularity_avg"].notna()].copy()
pop_df["wikipedia_id"] = pop_df["wikipedia_id"].astype("int64")

print(f"✓ Loaded {len(pop_df):,} Wikipedia pages with popularity data")
print(f"  Popularity range: {pop_df['popularity_avg'].min():.2f} - {pop_df['popularity_avg'].max():.2f}")
display(pop_df.head())

# Create ID lookup for faster matching
pop_lookup = pop_df.set_index("wikipedia_id")

Loading Wikipedia popularity data...
✓ Loaded 5,890,044 Wikipedia pages with popularity data
  Popularity range: 1.00 - 175763171.77


,wikipedia_id,wikipedia_title,popularity_avg,popularity_rank
0,11566748,Ossi Oikarinen,71.437500,2.957879e+06
1,37228152,2012–13 NBL Canada season,49.354167,3.259824e+06
2,30849763,Quercus iberica,93.708333,2.555272e+06
3,14734767,Fladnitz im Raabtal,12.791667,4.756272e+06
4,18662646,Jaszczołty,6.208333,5.296610e+06


In [3]:

# ── Helper functions ──────────────────────────────────────────────────────────

import json

def normalize_answers(raw):
    """Normalise any answer payload into a flat list of strings."""
    if not raw:
        return []
    if isinstance(raw, str):
        try:
            raw = json.loads(raw)
        except json.JSONDecodeError:
            return [raw]
    if isinstance(raw, dict):
        raw = [raw]
    if not isinstance(raw, list):
        return []
    out = []
    for entry in raw:
        if isinstance(entry, dict):
            val = entry.get("text") or entry.get("answer") or entry.get("value")
        elif isinstance(entry, (int, float)):
            val = str(entry)
        else:
            val = entry
        if val:
            out.append(val)
    return out


def extract_kilt_examples(ds, desc="Processing"):
    """Extract (question_id, question_text, answer_texts, wikipedia_id) from any KILT dataset."""
    data = []
    for ex in tqdm(ds, desc=desc):
        answers = [o["answer"] for o in (ex.get("output") or []) if o.get("answer")]
        wiki_ids = set()
        for o in (ex.get("output") or []):
            for prov in o.get("provenance") or []:
                try:
                    wiki_ids.add(int(prov["wikipedia_id"]))
                except (KeyError, ValueError, TypeError):
                    pass
        for wid in wiki_ids:
            data.append({
                "question_id": ex["id"],
                "question_text": ex["input"],
                "answer_texts": answers,
                "wikipedia_id": wid,
            })
    return pd.DataFrame(data)


def extract_triviaqa_examples(kilt_triviaqa_ds, trivia_qa_ds):
    """Extract TriviaQA examples with proper per-split question text mapping.

    KILT-TriviaQA only stores IDs — the question text must be mapped back from
    the original TriviaQA dataset on a per-split basis (as per KILT docs).
    """
    from datasets import load_dataset

    data = []
    for split in ["train", "validation", "test"]:
        if split not in kilt_triviaqa_ds or split not in trivia_qa_ds:
            continue

        # Build id → index map for this split
        triviaqa_map = {q_id: i for i, q_id in enumerate(trivia_qa_ds[split]["question_id"])}

        kilt_split = kilt_triviaqa_ds[split]
        for ex in tqdm(kilt_split, desc=f"  TriviaQA [{split}]"):
            kilt_id = ex["id"]
            if kilt_id not in triviaqa_map:
                continue

            orig = trivia_qa_ds[split][triviaqa_map[kilt_id]]
            question_text = orig["question"]
            answers = [orig["answer"]["value"]]
            # Also include any additional KILT answers
            for o in (ex.get("output") or []):
                if o.get("answer") and o["answer"] not in answers:
                    answers.append(o["answer"])

            wiki_ids = set()
            for o in (ex.get("output") or []):
                for prov in (o.get("provenance") or []):
                    try:
                        wiki_ids.add(int(prov["wikipedia_id"]))
                    except (KeyError, ValueError, TypeError):
                        pass

            for wid in wiki_ids:
                data.append({
                    "question_id": kilt_id,
                    "question_text": question_text,
                    "answer_texts": answers,
                    "wikipedia_id": wid,
                })

    return pd.DataFrame(data)


def merge_and_save(df, pop_df, output_path, name):
    """Merge with popularity lookup, reorder columns, and save to parquet."""
    if df.empty:
        print(f"  ⚠️  {name}: empty — skipping")
        return df
    df = df.copy()
    df["wikipedia_id"] = df["wikipedia_id"].astype("int64")
    merged = df.merge(pop_df, how="inner", on="wikipedia_id")
    merged = merged[[
        "question_id", "question_text", "answer_texts",
        "wikipedia_id", "wikipedia_title", "popularity_avg", "popularity_rank"
    ]]
    merged.to_parquet(output_path, index=False)
    pct = len(merged) / len(df) * 100
    print(f"  ✓ {name}: {len(df):,} pairs → {len(merged):,} matched ({pct:.1f}%) → {output_path.name}")
    return merged


print("✓ Helper functions defined")


✓ Helper functions defined


## 3. Process PopQA Dataset

In [4]:
print("="*60)
print("PROCESSING POPQA DATASET")
print("="*60)

popqa_ds = load_dataset(POPQA_PATH, split="test", cache_dir=CACHE_DIR)
print(f"Loaded {len(popqa_ds):,} questions")

popqa_data = []
for i, ex in enumerate(tqdm(popqa_ds, desc="Processing PopQA")):
    title = ex.get("s_wiki_title") or ex.get("subj")
    if not title:
        continue
    popqa_data.append({
        "question_id": ex.get("id", f"popqa_{i}"),
        "question_text": ex["question"],
        "answer_texts": normalize_answers(ex.get("possible_answers"))[:1],
        "wikipedia_title": title,
    })

popqa_df = pd.DataFrame(popqa_data)

# Title-based merge (PopQA has no wikipedia_id in the source)
popqa_merged = popqa_df.merge(pop_df, how="inner", on="wikipedia_title")
popqa_merged = popqa_merged[[
    "question_id", "question_text", "answer_texts",
    "wikipedia_id", "wikipedia_title", "popularity_avg", "popularity_rank"
]]
popqa_merged.to_parquet(POPQA_OUTPUT, index=False)

pct = len(popqa_merged) / len(popqa_df) * 100
print(f"  ✓ PopQA: {len(popqa_df):,} questions → {len(popqa_merged):,} matched ({pct:.1f}%) → {POPQA_OUTPUT.name}")
display(popqa_merged.head())

del popqa_ds, popqa_df
gc.collect()


PROCESSING POPQA DATASET


Repo card metadata block was not found. Setting CardData to empty.


Loaded 14,267 questions


Processing PopQA: 100%|██████████| 14267/14267 [00:00<00:00, 27684.69it/s]


  ✓ PopQA: 14,267 questions → 13,811 matched (96.8%) → popqa_with_popularity.parquet


,question_id,question_text,answer_texts,wikipedia_id,wikipedia_title,popularity_avg,popularity_rank
0,4222362,What is George Rankin's occupation?,[politician],16304924,George Rankin,70.770833,2.864337e+06
1,4725190,What is John Mayne's occupation?,[journalist],1098597,John Mayne,79.125000,2.780602e+06
2,4382392,What is Henry Feilden's occupation?,[politician],30108267,Henry Feilden (Conservative politician),17.895833,4.422461e+06
3,4822110,What is Kathy Saltzman's occupation?,[politician],23256856,Kathy Saltzman,40.270833,3.489112e+06
4,4011112,What is Eleanor Davis's occupation?,[cartoonist],27077682,Eleanor Davis,249.000000,1.664860e+06


0

## 4. Process KILT Datasets (NQ, TriviaQA, HotPotQA, FEVER, T-REx)

All five datasets share the same KILT schema — a single loop handles extraction, merging, and saving.


In [ ]:
print("="*60)
print("PROCESSING KILT DATASETS")
print("="*60)

# TriviaQA is handled separately below — KILT only stores IDs for it,
# not question text, so a per-split mapping back to the original dataset is required.
KILT_CONFIGS = [
    ("Natural Questions", NQ_PATH,       "nq",             NQ_OUTPUT),
    ("HotPotQA",          HOTPOTQA_PATH,  "hotpotqa",       HOTPOTQA_OUTPUT),
    ("FEVER",             FEVER_PATH,     "fever",          FEVER_OUTPUT),
    ("T-REx",             TREX_PATH,      "trex",           TREX_OUTPUT),
]

kilt_results = {}

for name, path, kilt_name, output_path in KILT_CONFIGS:
    print(f"\n── {name} ──")
    ds = load_dataset(path, name=kilt_name, split="train+validation+test", cache_dir=CACHE_DIR)
    print(f"  Loaded {len(ds):,} examples")
    df = extract_kilt_examples(ds, desc=f"  Extracting {name}")
    kilt_results[name] = merge_and_save(df, pop_df, output_path, name)
    del ds, df
    gc.collect()

# ── TriviaQA — requires per-split mapping back to original questions ──────────
print(f"\n── TriviaQA ──")
print("  Loading KILT TriviaQA (all splits)...")
kilt_triviaqa = load_dataset(TRIVIAQA_PATH, name="triviaqa_support_only", cache_dir=CACHE_DIR)

print("  Loading original TriviaQA for question text...")
trivia_qa = load_dataset("trivia_qa", "unfiltered.nocontext", cache_dir=CACHE_DIR)

triviaqa_df = extract_triviaqa_examples(kilt_triviaqa, trivia_qa)
print(f"  Extracted {len(triviaqa_df):,} question-document pairs")
kilt_results["TriviaQA"] = merge_and_save(triviaqa_df, pop_df, TRIVIAQA_OUTPUT, "TriviaQA")

del kilt_triviaqa, trivia_qa, triviaqa_df
gc.collect()

print("\n✓ All KILT datasets processed")


PROCESSING KILT DATASETS

── Natural Questions ──
  Loaded 91,653 examples


  Extracting Natural Questions:  40%|████      | 36713/91653 [00:01<00:02, 23517.71it/s]

## 5. Summary Statistics


In [ ]:
print("="*60)
print("FINAL SUMMARY")
print("="*60)

all_outputs = {
    "PopQA":              POPQA_OUTPUT,
    "Natural Questions":  NQ_OUTPUT,
    "TriviaQA":           TRIVIAQA_OUTPUT,
    "HotPotQA":           HOTPOTQA_OUTPUT,
    "FEVER":              FEVER_OUTPUT,
    "T-REx":              TREX_OUTPUT,
}

rows = []
for name, path in all_outputs.items():
    if path.exists():
        df = pd.read_parquet(path)
        rows.append({
            "Dataset":           name,
            "Rows":              f"{len(df):,}",
            "Unique questions":  f"{df['question_id'].nunique():,}",
            "Unique wiki pages": f"{df['wikipedia_id'].nunique():,}",
            "Pop avg (median)":  f"{df['popularity_avg'].median():.1f}",
        })
        del df
    else:
        rows.append({"Dataset": name, "Rows": "❌ not generated"})

display(pd.DataFrame(rows).set_index("Dataset"))


FINAL SUMMARY


,Rows,Unique questions,Unique wiki pages,Pop avg (median)
Dataset,,,,
PopQA,"13,811","13,811","11,852",903.0
Natural Questions,"81,533","79,756","39,402",18366.6
TriviaQA,"98,386","58,215","38,389",32331.8
HotPotQA,"148,414","74,257","88,222",2952.1
FEVER,"94,367","81,701","8,436",67359.1
T-REx,"2,839,105","2,286,338","1,513,556",73.0


## 6. Upload Enriched QA Files to HuggingFace

Authenticate via `HUGGINGFACE_TOKEN` and push each parquet artifact to the configured repo so downstream consumers can access the enriched QA datasets.


In [ ]:
from huggingface_hub import login, HfApi
from datasets import Dataset, DatasetDict
from io import BytesIO
import pandas as pd

if not HUGGINGFACE_TOKEN:
    raise RuntimeError("HUGGINGFACE_TOKEN is not set; cannot upload datasets.")

print("Logging in to HuggingFace...")
login(token=HUGGINGFACE_TOKEN)

# Upload README
print("Uploading README.md...")
api = HfApi()
api.upload_file(
    path_or_fileobj=BytesIO(README_TEXT.strip().encode("utf-8")),
    path_in_repo="README.md",
    repo_id=HUGGINGFACE_UPLOAD_REPO,
    repo_type="dataset",
    token=HUGGINGFACE_TOKEN,
)

for label, (subset_name, dataset_path) in HUGGINGFACE_UPLOAD_SUBSETS.items():
    if not dataset_path.exists():
        print(f"Skipping {label}: File not found.")
        continue

    print(f"Processing {subset_name}...")
    df = pd.read_parquet(dataset_path)
    df["question_id"] = df["question_id"].astype(str)
    
    # 90/10 Train/Test split
    df = df.sample(frac=1, random_state=42).reset_index(drop=True)
    split_idx = int(len(df) * 0.9)
    train_df, test_df = df.iloc[:split_idx], df.iloc[split_idx:]
    
    ds_dict = DatasetDict({
        "train": Dataset.from_pandas(train_df, preserve_index=False),
        "test": Dataset.from_pandas(test_df, preserve_index=False)
    })
    
    ds_dict.push_to_hub(
        repo_id=HUGGINGFACE_UPLOAD_REPO,
        config_name=subset_name,
        token=HUGGINGFACE_TOKEN
    )
    print(f"✓ Pushed {subset_name} configuration")

print(f"\nUpload complete: https://huggingface.co/datasets/{HUGGINGFACE_UPLOAD_REPO}")

Logging in to HuggingFace...
Uploading README.md...
Processing pop_qa...


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/13 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

...

Creating parquet from Arrow format:   0%|          | 0/284 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

✓ Pushed trex configuration

Upload complete: https://huggingface.co/datasets/Cyro1/popularity-enriched-qa-datasets
